# Beyond Development: Explaining Climate Resilience Across Indian Districts — Review 2

**Research question.** Why do some economically developed Indian districts remain highly climate-vulnerable,
while some less-developed districts are resilient?

| Section | Content | Owner |
|---|---|---|
| 0 | Setup: Drive, repository, packages, tests | Sehjodh Grewal |
| 1 | Data download: IMD 1991–2020 to Google Drive | Sehjodh Grewal |
| 2 | Preprocessing + Table 1 + preprocessing error metrics | Sehjodh Grewal |
| 3 | Big Data platform: Hadoop (HDFS + YARN), Hive, Spark | Deepankar |
| 4 | HDFS load, listing, Table 2 | Deepankar |
| 5 | MapReduce on YARN: local test, job, timings | Tanmay Singh |
| 6 | Hive: 30-year baseline + join, check against pandas | Tanmay Singh |
| 7 | Spark k-means climate regimes + Table 3 | Tanmay Singh |
| 8 | A-CPN (proposed model): architecture, parameters, Table 4 | Devanshu Kumar |
| 9 | Comparison models: Spark MLlib (LR, RF, GBT, Linear SVM) + MLP | Devanshu Kumar |
| 10 | Tables 5 and 6: comparison, metrics, ablation, climate push, preprocessing before/after | Devanshu Kumar |
| 11 | Visualisations (only after every table) | each owner |
| 12 | Build the slides; appendix to show code | Tanmay Singh |

**Fast path:** to show only the models, run section 0 and then sections 8–12; they use the committed
`data/processed/analysis_table.csv`. Single-node pseudo-distributed Hadoop (replication 1): no multi-machine
speed-up is claimed.

## 0. Setup  *(owner: Sehjodh Grewal)*

In [ ]:
import os, subprocess, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = '/content/drive/MyDrive/climate_project'      # Colab wipes its disk: persistent data lives on Drive
    REPO_DIR, IMD_DIR = f'{PROJECT}/repo', f'{PROJECT}/imd'
    os.makedirs(PROJECT, exist_ok=True)
    if not os.path.isdir(REPO_DIR):
        subprocess.run(['git', 'clone', 'https://github.com/deepankarkumar1310/hadoop-ml-algorithm-.git', REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    print(subprocess.run(['git', 'pull', '--ff-only'], capture_output=True, text=True).stdout.strip())
else:
    os.chdir(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())
    IMD_DIR = 'data/imd'
START, END = 1991, 2020
os.environ.update(IMD_DIR=IMD_DIR, START=str(START), END=str(END))
print(os.getcwd())

In [ ]:
!python3 -m pip -q install -r requirements.txt
!python3 -c "import torch, pyspark, imdlib; print('torch', torch.__version__, '| pyspark', pyspark.__version__)"
!python3 -m pytest tests -q

## 1. Dataset and IMD download  *(owner: Sehjodh Grewal)*

| Dataset | Source | Year | Format | Role |
|---|---|---|---|---|
| Climate Vulnerability Index + indicators (699 × 79) | NITI Aayog NDAP 7160 | 2021 | CSV | outcome + 2 development variables |
| Primary Census Abstract (640 districts) | Census of India | 2011 | CSV | 3 development variables (2011 baseline) |
| District boundaries (641 polygons) | DataMeet | 2011 | shapefile | grid → district weights |
| Daily gridded rainfall 0.25° | IMD Pune | 1991–2020 | binary .grd | hazard |
| Daily gridded Tmax / Tmin 1° | IMD Pune | 1991–2020 | binary .grd | hazard |

The IMD files (~848 MB) are downloaded to Google Drive and are never committed to GitHub.

In [ ]:
!python3 scripts/download_imd.py --imd-dir "$IMD_DIR" --start $START --end $END

## 2. Preprocessing and preprocessing results  *(owner: Sehjodh Grewal)*

P1 Census rates · P2 NDAP↔Census matching · P3 development score and quadrants · P4 grid→district area weights ·
P5 district-day table (missing stays missing) · P6 Chennai orientation gate · P7 standardisation on training folds only.

In [ ]:
!python3 scripts/census_vars.py | tail -8
!python3 scripts/match_districts.py | tail -8
!python3 scripts/build_integrated_table.py | tail -5
!python3 scripts/build_district_day.py --imd-dir "$IMD_DIR" --out-dir data/staging/district_day --start $START --end $END

In [ ]:
# TABLE 1 + preprocessing error metrics (each step checked against a reference)
import json, pandas as pd
!python3 scripts/preprocessing_metrics.py > /dev/null
pm = json.load(open('results/preprocessing_metrics.json'))
ar = json.load(open('results/analysis_results.json'))
t1 = pd.DataFrame([
    ('Census 2011 districts', 640), ('Matched to NDAP', pm['matching']['matched']),
    ('With vulnerability index', ar['preprocessing']['with_index']),
    ('Complete (all 13 features)', pm['final_table']['rows']),
    ('Census population (official 1,210,854,977)', f"{pm['census']['population']:,}"),
    ('National literacy (official 72.98%)', f"{pm['census']['literacy_pct']:.4f}%"),
    ('Class balance High / Low', f"{pm['final_table']['class_high']} / {pm['final_table']['class_low']}"),
    ('NDAP columns >50% empty', f"{pm['ndap']['cols_over_half_empty']} of {pm['ndap']['cols']}"),
], columns=['Stage / check', 'Value'])
display(t1)
display(pd.DataFrame(pm['feature_signal_auc'].items(), columns=['feature', 'single-feature ROC-AUC']))
pm

## 3. Big Data platform  *(owner: Deepankar)*

| Tool | Where | What it does | Why chosen |
|---|---|---|---|
| **HDFS** | `hadoop/setup_hadoop.sh`, `hadoop/load_hdfs.sh` | Stores raw, staging and processed zones | 128 MB blocks for parallel jobs; read directly by MapReduce, Hive, Spark |
| **YARN** | `hadoop/setup_hadoop.sh` | Allocates CPU/memory and runs MapReduce tasks | Hadoop's standard scheduler |
| **Hive** | `hive/setup_hive.sh`, `hive/project.hql` | SQL over HDFS: 30-year averages + join | Aggregation and joins are SQL tasks |
| **Spark** | `spark/setup_spark.sh`, `spark/*.py` | k-means climate regimes; MLlib baselines | In-memory, fast for iterative algorithms |

Java 8 for everything (Hive 3.1.3 fails on Java 11). Colab has no SSH, so every daemon is started directly.

In [ ]:
def load_env(script):
    out = subprocess.run(['bash', '-c', f'source {script} > /tmp/setup.log 2>&1 && env'], capture_output=True, text=True)
    print(open('/tmp/setup.log').read()[-1500:])
    for line in out.stdout.splitlines():
        k, _, v = line.partition('=')
        if k in ('JAVA_HOME', 'HADOOP_HOME', 'HADOOP_CONF_DIR', 'PATH', 'HIVE_HOME', 'HADOOP_STREAMING_JAR',
                 'HDFS_NAMENODE_USER', 'HDFS_DATANODE_USER', 'YARN_RESOURCEMANAGER_USER', 'YARN_NODEMANAGER_USER',
                 'PYSPARK_PYTHON', 'SPARK_LOCAL_IP'):
            os.environ[k] = v

load_env('hadoop/setup_hadoop.sh')
!hadoop version | head -1
!jps
!hdfs dfsadmin -report | head -12

## 4. HDFS load and Table 2  *(owner: Deepankar)*

In [ ]:
!bash hadoop/load_hdfs.sh "$IMD_DIR" data/staging/district_day
!hdfs dfs -put -f data/processed/analysis_table.csv /project/processed/
!hdfs dfs -ls /project /project/raw /project/raw/imd
!hdfs dfs -du -s -h /project/raw /project/staging /project/processed /project

## 5. MapReduce on YARN  *(owner: Tanmay Singh)*

**Map:** each district-day row → key (censuscode, year). **Reduce:** sort the days by date and compute 7 yearly
indices (rain total, heavy-rain days, max 5-day rain, longest dry spell, mean Tmax, hot days, tropical nights).
Incomplete years are left blank (never 0); malformed rows go to a Hadoop counter.

In [ ]:
# local test of the same mapper/reducer
!head -2000 data/staging/district_day/district_day_${END}.csv | python3 mapreduce/mapper.py | sort | python3 mapreduce/reducer.py 2>/dev/null | head -3
!bash hadoop/run_mapreduce.sh /project/staging/district_day /project/processed/district_year 2 2>&1 | grep -E "Map input records|Reduce output records|bad_rows|completed successfully|ERROR"
!hdfs dfs -getmerge /project/processed/district_year data/processed/district_year_indices.csv
!wc -l data/processed/district_year_indices.csv

In [ ]:
# scalability: 1 / 10 / 30 years, 2 repeats each (single node)
!python3 scripts/time_mapreduce.py --years 1 10 30 --repeats 2
!cat results/scalability_timings.csv
# preprocessing error checks that need the raw grids: MapReduce vs pandas, area-weighted vs nearest-cell
!python3 scripts/colab_checks.py --imd-dir "$IMD_DIR" --year $END

## 6. Hive: 30-year baseline + join, checked against pandas  *(owner: Tanmay Singh)*

In [ ]:
load_env('hive/setup_hive.sh')
!cd /opt/hive-metastore && hive --hivevar start=$START --hivevar end=$END -f $OLDPWD/hive/project.hql 2>/dev/null
!hdfs dfs -getmerge /project/hive/warehouse/project.db/integrated_district data/processed/integrated_district_hive.csv
!python3 scripts/build_climate_baseline.py --start $START --end $END
!python3 scripts/join_climate.py --check-hive data/processed/integrated_district_hive.csv
!cat results/hive_check.json

## 7. Spark k-means climate regimes + Table 3  *(owner: Tanmay Singh)*

In [ ]:
load_env('spark/setup_spark.sh')
!python3 spark/kmeans_regimes.py --input hdfs://localhost:9000/project/processed/analysis_table.csv --output hdfs://localhost:9000/project/processed/climate_regimes 2>&1 | grep -E "k=|chosen"
import pandas as pd, json
km = json.load(open('results/kmeans_regimes.json'))
t = pd.read_csv('results/scalability_timings.csv')
display(pd.DataFrame([
    ('MapReduce', f"{int(t.map_input_records.max()):,} district-day rows", f"{int(t.reduce_output_records.max()):,} district-years",
     f"{t[t.years == t.years.max()].runtime_s.mean():.0f} s"),
    ('Hive baseline + join', 'district-years + development table', '469-district table', 'checked against pandas'),
    ('Spark k-means', '8 hazard variables', f"{km['k']} climate regimes", f"silhouette {km['silhouette']:.3f}"),
], columns=['Job', 'Input', 'Output', 'Result']))

## 8. Proposed model: A-CPN (Attentive Climate Push Network)  *(owner: Devanshu Kumar)*

`logit = development baseline (linear, 5 features) + climate push (attention → KAN → residual MLP, 8 hazard features)`.
MC Dropout gives the uncertainty of each district's push; split conformal prediction gives 90% prediction sets.
The vulnerability index is only the target, never an input.

In [ ]:
import sys; sys.path.insert(0, '.')
from models.acpn import ACPN
net = ACPN()
print(net)
print('trainable parameters:', sum(p.numel() for p in net.parameters() if p.requires_grad))
# True = retrain every model here (~1 hour on 2 vCPU, mostly the Spark baselines);
# False = show the committed results in results/ (produced by the same scripts)
RERUN_MODELS = True
if RERUN_MODELS:
    !python3 scripts/train_acpn.py
print(open('results/table4_model_card.json').read())

## 9. Comparison models: Spark MLlib (LR, RF, GBT, Linear SVM) and a standard MLP  *(owner: Devanshu Kumar)*

In [ ]:
if RERUN_MODELS:
    !python3 spark/train_baselines.py 2>&1 | grep -E "folds done|districts read"
    !python3 scripts/train_mlp.py
    !python3 scripts/train_acpn.py --raw        # preprocessing before/after experiment
    !python3 scripts/train_acpn.py --ablation

## 10. Tables 5 and 6  *(owner: Devanshu Kumar)*

In [ ]:
!python3 scripts/evaluate.py > /dev/null
t5 = pd.read_csv('results/table5_comparison.csv')
print('TABLE 5 - comparison (mean over 25 folds; LOSO = leave-one-state-out)')
display(t5[['model', 'tool', 'roc_auc', 'roc_auc_sd', 'bal_acc', 'accuracy', 'f1', 'loso_auc', 'train_time']].round(3))
print('Detailed metrics'); display(pd.read_csv('results/classifier_metrics_full.csv').round(3))
print('Confusion matrices (summed over 25 folds)'); display(pd.read_csv('results/confusion_matrices.csv'))
print('95% CI and corrected paired t-test vs A-CPN'); display(pd.read_csv('results/significance.csv').round(4))
print('TABLE 6a - ablation'); display(pd.read_csv('results/table6_ablation.csv').round(3))
print('Preprocessing before vs after (raw vs preprocessed features)'); display(pd.read_csv('results/preprocessing_before_after.csv').round(3))
print('TABLE 6b - climate push'); print(open('results/table6_climate_push.json').read()[:3000])

## 11. Visualisations (only after every table)  *(each owner)*

In [ ]:
!python3 scripts/make_figures.py > /dev/null
from IPython.display import Image, display
for f in ['prep_retention', 'prep_quality_before_after', 'prep_feature_scaling', 'prep_model_gain', 'prep_feature_signal',
          'prep_correlation', 'hdfs_storage', 'mapreduce_scalability', 'kmeans_silhouette', 'roc_curves', 'comparison_bars',
          'fold_boxplots', 'confusion_matrices', 'calibration', 'loso_by_state', 'acpn_training_curve', 'ablation',
          'attention_importance', 'climate_push_by_quadrant', 'climate_push_map']:
    print(f); display(Image(f'figures/{f}.png'))

## 12. Slides and appendix  *(owner: Tanmay Singh)*

In [ ]:
!python3 scripts/build_deck.py

In [ ]:
# appendix: show any source file during the review
!cat mapreduce/mapper.py mapreduce/reducer.py
!cat hive/project.hql
!cat models/acpn.py